In [1]:
# Install required libraries (useful when running in Kaggle / fresh environments)
!pip install -q monai nibabel scipy

import os
import torch
import numpy as np
import nibabel as nib
import scipy.ndimage as ndi

# MONAI is used for medical imaging workflows(transforms,datasets, models)
from monai.transforms import *
from monai.data import Dataset, DataLoader
from monai.networks.nets import UNet
from monai.losses import DiceLoss

import torch.nn as nn

# Used for splitting dataset into train/validation/test sets
from sklearn.model_selection import train_test_split

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 28.3 MB/s eta 0:00:00


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-03-29 03:47:28.498996: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774756048.707619      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774756048.767505      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774756049.295316      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774756049.295363      24 computation_placer.cc:1

In [2]:
# ---------------------------------------------------------
# Dataset Path
# ---------------------------------------------------------
# Base directory containing CT volumes and corresponding masks
# Expected structure:
# heart_dataset/
#   ├── images/
#   ├── masks/
# ---------------------------------------------------------

base = "/kaggle/input/datasets/vanshoberoi03/heart-dataset/heart_dataset"


# ---------------------------------------------------------
# Collect valid image-mask pairs
# ---------------------------------------------------------
# We iterate through all image files and only keep those
# samples for which a corresponding mask exists.
# This ensures consistency during training.
# ---------------------------------------------------------

images = sorted(os.listdir(base + "/images"))

data = []

for img in images:
    img_path = base + "/images/" + img
    mask_path = base + "/masks/" + img

    # Only include samples with valid mask annotations
    if os.path.exists(mask_path):
        data.append({
            "image": img_path,
            "label": mask_path,
            "name": img.replace(".nii", "")
        })

print("Total valid samples:", len(data))

Total valid samples: 50


In [3]:
# ---------------------------------------------------------
# Train / Validation / Test Split
# ---------------------------------------------------------
# The dataset is split into:
# - 80% training
# - 10% validation
# - 10% testing
#
# A fixed random seed is used to ensure reproducibility.
# ---------------------------------------------------------

train_files, temp_files = train_test_split(
    data, test_size=0.2, random_state=42
)

val_files, test_files = train_test_split(
    temp_files, test_size=0.5, random_state=42
)

In [4]:
# ---------------------------------------------------------
# Preprocessing and Augmentation Pipeline
# ---------------------------------------------------------
# These transforms prepare the CT volumes for training:
# - Ensure consistent spatial dimensions
# - Apply HU-based intensity scaling
# - Introduce augmentation for better generalization
# ---------------------------------------------------------

train_transforms = Compose([

    # Load image and corresponding segmentation mask
    LoadImaged(keys=["image", "label"]),

    # Convert to channel-first format: (C, H, W, D)
    EnsureChannelFirstd(keys=["image", "label"]),

    # Resize all volumes to a fixed spatial resolution
    # This ensures consistency across the dataset
    ResizeD(keys=["image", "label"], spatial_size=(128,128,128)),

    # HU-based intensity windowing:
    # - Clips values to a relevant cardiac range
    # - Normalizes intensities to [0, 1]
    # This helps highlight calcified regions in CT scans
    ScaleIntensityRanged(
        keys=["image"],
        a_min=-100,
        a_max=400,
        b_min=0.0,
        b_max=1.0,
        clip=True
    ),

    # Data augmentation to improve robustness
    # Random flips across all spatial axes
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),

    # Random 90-degree rotations to simulate orientation variability
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
])


# ---------------------------------------------------------
# Validation / Test Transforms
# ---------------------------------------------------------
# Same preprocessing as training but WITHOUT augmentation
# Ensures consistent evaluation
# ---------------------------------------------------------

val_transforms = Compose([

    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),

    ResizeD(keys=["image", "label"], spatial_size=(128,128,128)),

    # Same intensity scaling for consistency during evaluation
    ScaleIntensityRanged(
        keys=["image"],
        a_min=-100,
        a_max=400,
        b_min=0.0,
        b_max=1.0,
        clip=True
    )
])

In [5]:
# ---------------------------------------------------------
# Dataset and DataLoader Setup
# ---------------------------------------------------------
# MONAI Dataset wraps the file paths and applies the defined transforms.
# DataLoader enables efficient batching and iteration during training.
# ---------------------------------------------------------

# Create dataset objects with corresponding preprocessing pipelines
train_ds = Dataset(train_files, transform=train_transforms)
val_ds   = Dataset(val_files, transform=val_transforms)
test_ds  = Dataset(test_files, transform=val_transforms)


# Create DataLoaders for batch-wise processing
# Batch size is set to 1 due to high memory requirements of 3D volumes
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)

# Validation and test loaders do not require shuffling
val_loader   = DataLoader(val_ds, batch_size=1)
test_loader  = DataLoader(test_ds, batch_size=1)

In [6]:
# ---------------------------------------------------------
# Device Configuration
# ---------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ---------------------------------------------------------
# Model Definition: 3D U-Net
# ---------------------------------------------------------
# A 3D U-Net is used for volumetric segmentation as it captures
# both local details and global spatial context effectively.
#
# The encoder extracts hierarchical features, while the decoder
# reconstructs spatial resolution using skip connections.
# ---------------------------------------------------------

model = UNet(
    spatial_dims=3,          # 3D data (CT volumes)
    in_channels=1,           # single-channel grayscale input
    out_channels=1,          # binary segmentation output

    # Feature channels at each level (increasing depth)
    channels=(16, 32, 64, 128, 256),

    # Downsampling strides for encoder
    strides=(2, 2, 2, 2),

    # Residual units help stabilize training and improve gradients
    num_res_units=2,
).to(device)

In [7]:
# ---------------------------------------------------------
# Loss Functions
# ---------------------------------------------------------
# Dice Loss focuses on overlap between prediction and ground truth,
# making it well-suited for segmentation tasks with class imbalance.
#
# BCE Loss provides stable pixel-wise supervision.
# Combining both helps balance global overlap and local accuracy.
# ---------------------------------------------------------

dice_loss = DiceLoss(sigmoid=True)
bce_loss = nn.BCEWithLogitsLoss()


# Hybrid loss combining Dice and BCE
# Weighted combination helps stabilize training while optimizing overlap
def combined_loss(pred, target):
    return 0.7 * dice_loss(pred, target) + 0.3 * bce_loss(pred, target)


# ---------------------------------------------------------
# Optimizer
# ---------------------------------------------------------
# Adam optimizer is used for adaptive learning and faster convergence
# ---------------------------------------------------------

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


# ---------------------------------------------------------
# Learning Rate Scheduler
# ---------------------------------------------------------
# Reduces learning rate when validation performance plateaus
# Helps refine training and avoid overfitting
# ---------------------------------------------------------

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',      # maximize validation Dice score
    patience=5,      # wait before reducing LR
    factor=0.5       # reduce LR by half
)

In [8]:
# ---------------------------------------------------------
# Training Loop
# ---------------------------------------------------------
# The model is trained for multiple epochs with validation
# after each epoch to monitor generalization performance.
#
# Best model is saved based on validation Dice score.
# ---------------------------------------------------------

epochs = 100
best_dice = 0

for epoch in range(epochs):

    # -------------------- TRAINING --------------------
    model.train()
    total_loss = 0

    for batch in train_loader:

        # Move data to device (GPU/CPU)
        img = batch["image"].to(device)
        mask = batch["label"].to(device)

        # Forward pass
        pred = model(img)

        # Compute loss
        loss = combined_loss(pred, mask)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # -------------------- VALIDATION --------------------
    model.eval()
    val_dice = []

    with torch.no_grad():
        for batch in val_loader:

            img = batch["image"].to(device)
            mask = batch["label"].to(device)

            # Apply sigmoid to convert logits → probabilities
            pred = torch.sigmoid(model(img))

            # Thresholding to obtain binary mask
            pred = (pred > 0.4).float()

            # Dice computation (per volume)
            intersection = (pred * mask).sum(dim=(1,2,3,4))
            dice = (2 * intersection) / (
                pred.sum(dim=(1,2,3,4)) + mask.sum(dim=(1,2,3,4)) + 1e-8
            )

            val_dice.extend(dice.cpu().numpy())

    avg_dice = np.mean(val_dice)

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Val Dice: {avg_dice:.4f}")

    # -------------------- MODEL CHECKPOINT --------------------
    # Save best model based on validation Dice score
    if avg_dice > best_dice:
        best_dice = avg_dice
        torch.save(model.state_dict(), "best_model.pth")

    # -------------------- LR SCHEDULER --------------------
    scheduler.step(avg_dice)

Epoch 1, Loss: 31.5712, Val Dice: 0.2024
Epoch 2, Loss: 29.6730, Val Dice: 0.2032
Epoch 3, Loss: 28.7828, Val Dice: 0.2033
Epoch 4, Loss: 28.5067, Val Dice: 0.2032
Epoch 5, Loss: 28.1241, Val Dice: 0.2032
Epoch 6, Loss: 27.7345, Val Dice: 0.2031
Epoch 7, Loss: 27.5174, Val Dice: 0.2032
Epoch 8, Loss: 27.2737, Val Dice: 0.2033
Epoch 9, Loss: 27.0918, Val Dice: 0.2037
Epoch 10, Loss: 27.0102, Val Dice: 0.2039
Epoch 11, Loss: 26.6408, Val Dice: 0.2040
Epoch 12, Loss: 26.4715, Val Dice: 0.2036
Epoch 13, Loss: 26.2517, Val Dice: 0.2041
Epoch 14, Loss: 26.0474, Val Dice: 0.2043
Epoch 15, Loss: 25.9510, Val Dice: 0.2058
Epoch 16, Loss: 25.6924, Val Dice: 0.2070
Epoch 17, Loss: 25.4577, Val Dice: 0.2064
Epoch 18, Loss: 25.3292, Val Dice: 0.2066
Epoch 19, Loss: 25.1213, Val Dice: 0.2143
Epoch 20, Loss: 25.0417, Val Dice: 0.2151
Epoch 21, Loss: 24.7409, Val Dice: 0.2179
Epoch 22, Loss: 24.5690, Val Dice: 0.2388
Epoch 23, Loss: 24.3448, Val Dice: 0.2924
Epoch 24, Loss: 24.5027, Val Dice: 0.2598
E

In [9]:
# ---------------------------------------------------------
# Post-processing: Largest Connected Component
# ---------------------------------------------------------
# In medical segmentation, models often predict small isolated
# regions (false positives) due to noise or intensity similarity.
#
# Since the heart region is expected to be spatially continuous,
# we retain only the largest connected component.
# This helps eliminate spurious detections (e.g., ribs, vertebrae).
# ---------------------------------------------------------

def largest_component(mask):
    
    # Label connected regions in the binary mask
    labeled, num = ndi.label(mask)

    # If no connected regions exist, return original mask
    if num == 0:
        return mask

    # Compute size of each connected component
    sizes = ndi.sum(mask, labeled, range(1, num + 1))

    # Identify the largest component
    largest = (sizes.argmax() + 1)

    # Return only the largest component
    return (labeled == largest).astype(mask.dtype)

In [10]:
# ---------------------------------------------------------
# Load Best Model
# ---------------------------------------------------------
# Load the best-performing model (based on validation Dice)
# ---------------------------------------------------------

model.load_state_dict(torch.load("best_model.pth"))
model.eval()


# ---------------------------------------------------------
# Threshold Optimization
# ---------------------------------------------------------
# Instead of using a fixed threshold (e.g., 0.5),
# we search for an optimal threshold that maximizes Dice score
# on the validation set.
#
# This is important because CAC regions are sparse, and
# lower thresholds often improve sensitivity.
# ---------------------------------------------------------

def find_best_threshold(model, val_loader, device):

    # Candidate thresholds to evaluate
    thresholds = [0.3, 0.35, 0.4, 0.45, 0.5]

    best_t, best_score = 0, 0

    with torch.no_grad():
        for t in thresholds:

            dice_scores = []

            for batch in val_loader:

                img = batch["image"].to(device)
                mask = batch["label"].to(device)

                # Convert logits → probabilities
                pred = torch.sigmoid(model(img))

                # Apply threshold
                pred = (pred > t).float()

                # Dice computation
                intersection = (pred * mask).sum(dim=(1,2,3,4))
                dice = (2 * intersection) / (
                    pred.sum(dim=(1,2,3,4)) + mask.sum(dim=(1,2,3,4)) + 1e-8
                )

                dice_scores.extend(dice.cpu().numpy())

            avg = np.mean(dice_scores)

            # Track best threshold
            if avg > best_score:
                best_score = avg
                best_t = t

    return best_t


# ---------------------------------------------------------
# Find Optimal Threshold
# ---------------------------------------------------------

best_threshold = find_best_threshold(model, val_loader, device)
print("Best Threshold:", best_threshold)

Best Threshold: 0.5


In [11]:
# ---------------------------------------------------------
# Final Evaluation on Test Set
# ---------------------------------------------------------
# Evaluate model performance on unseen data using Dice score.
# Includes thresholding + post-processing for realistic results.
# ---------------------------------------------------------

dice_scores = []

with torch.no_grad():
    for batch in test_loader:

        img = batch["image"].to(device)
        mask = batch["label"].to(device)

        # Convert logits → probabilities
        pred = torch.sigmoid(model(img))

        # Apply optimal threshold (learned from validation set)
        pred = (pred > best_threshold).float()

        # Convert to numpy for post-processing
        pred_np = pred.cpu().numpy()[0, 0]

        # Remove small false positives using largest component
        pred_np = largest_component(pred_np)

        # Convert back to tensor
        pred = torch.tensor(pred_np).unsqueeze(0).unsqueeze(0).to(device)

        # Dice score computation
        intersection = (pred * mask).sum()
        dice = (2 * intersection) / (pred.sum() + mask.sum() + 1e-8)

        dice_scores.append(dice.item())

# Final metric
print("Final Test Dice:", np.mean(dice_scores))

Final Test Dice: 0.9168105244636535


In [12]:
# ---------------------------------------------------------
# Inference Time Measurement
# ---------------------------------------------------------
# Measure average time taken to process one CT volume.
# This is important for real-world deployment where
# both accuracy and efficiency matter.
# ---------------------------------------------------------

import time

model.eval()
times = []

with torch.no_grad():
    for batch in test_loader:

        img = batch["image"].to(device)

        # Start timing
        start = time.time()

        # Forward pass
        pred = torch.sigmoid(model(img))

        # Apply optimal threshold
        pred = (pred > best_threshold).float()

        # Post-processing (remove small false positives)
        pred_np = pred.cpu().numpy()[0, 0]
        pred_np = largest_component(pred_np)

        # End timing
        end = time.time()

        times.append(end - start)

# Compute average inference time
avg_time = sum(times) / len(times)

print("Average Inference Time per volume:", avg_time, "seconds")

Average Inference Time per volume: 0.07956924438476562 seconds


In [13]:
# ---------------------------------------------------------
# Saving Predictions as NIfTI Files
# ---------------------------------------------------------
# This step converts model outputs into .nii format,
# which is the standard format used in medical imaging.
#
# Preserving affine ensures correct spatial alignment.
# ---------------------------------------------------------

output_dir = "/kaggle/working/predictions"
os.makedirs(output_dir, exist_ok=True)

model.eval()

with torch.no_grad():
    for batch in test_loader:

        img = batch["image"].to(device)
        name = batch["name"][0]

        # Forward pass
        pred = torch.sigmoid(model(img))

        # Apply optimal threshold
        pred = (pred > best_threshold).float()

        # Convert to numpy
        pred_np = pred.cpu().numpy()[0, 0]

        # Post-processing (retain largest component)
        pred_np = largest_component(pred_np)

        # -------------------------------------------------
        # Preserve spatial metadata (affine matrix)
        # -------------------------------------------------
        if "image_meta_dict" in batch:
            affine = batch["image_meta_dict"]["affine"][0].numpy()
        else:
            affine = np.eye(4)  # fallback if metadata not available

        # Create NIfTI object
        nii = nib.Nifti1Image(pred_np.astype(np.uint8), affine)

        # Save prediction
        save_path = os.path.join(output_dir, f"{name}_pred.nii")
        nib.save(nii, save_path)

print("Predictions saved at:", output_dir)

Predictions saved at: /kaggle/working/predictions
